# Day 11 — Modules, Packages, and the requests Library

======================================================
Module 01: Python + Async + FastAPI for LLM Engineering
Vidya Sankalp | Applied GenAI Engineering

WHAT THIS FILE COVERS:
  PART A — Modules vs Packages (Python project structure)
    - What is a module? (a .py file)
    - What is a package? (a folder with __init__.py)
    - How imports work (absolute vs relative)
    - Why project structure matters for LLM apps

  PART B — requests library (synchronous HTTP client)
    - GET and POST requests
    - Sending JSON to an LLM API
    - Handling responses and errors
    - requests vs httpx (async) — which to use when

  WHY THIS MATTERS FOR LLM ENGINEERING:
  - Every LLM API call is an HTTP POST request
  - Understanding imports is essential before building FastAPI apps
  - requests is the simplest way to call an API from a script
  - httpx (async, from Day 09) is what you use inside FastAPI endpoints


In [ ]:

import json
import logging
import os
import time
from pathlib import Path

import requests       # pip install requests
from dotenv import load_dotenv

load_dotenv()
log = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")




## PART A — MODULE vs PACKAGE

═════════════════════════════════════════════════════════════════════════════


## A MODULE is a single .py file.

This file (day11_modules_packages_requests.py) IS a module.
day01_setup_variables.py is a module.
json, os, logging — these are all modules (built into Python's stdlib).
To import a module:
import json                    # import the whole module → use json.loads()
from json import loads         # import one name → use loads()
from json import loads, dumps  # import multiple names
from json import *             # import everything (avoid — pollutes namespace)
import json as j               # alias → use j.loads()


In [ ]:

# Example: importing the day01 module to reuse its functions
# (Commented out here since the path would need to be in sys.path)
#   from modules.day01_setup_variables import build_customer_support_prompt
#   from modules.day05_exception_handling_logging import LLMRateLimitError




## A PACKAGE is a FOLDER containing:

1. An __init__.py file (can be empty)
2. One or more .py modules
3. Optionally, sub-packages (folders with their own __init__.py)
EXAMPLE — The fastapi_app/ folder in this project IS a package:
fastapi_app/              ← this is a package
├── __init__.py           ← makes it a package (can be empty)
├── main.py               ← module inside the package
├── models.py             ← module inside the package
├── database.py           ← module inside the package
└── routers/              ← sub-package
├── __init__.py       ← makes routers/ a sub-package
├── customers.py      ← module in the sub-package
└── products.py       ← module in the sub-package
To import from a package:
from fastapi_app.models import CustomerRecord   # absolute import
from fastapi_app.routers import customers       # import sub-module
from .models import CustomerRecord              # relative import (inside the package)


## __init__.py tells Python: "this folder is a package, not just a directory."

It runs automatically when the package is first imported.
Uses of __init__.py:
1. Empty: just marks the folder as a package (most common)
2. Re-export: expose selected names so imports are shorter
# In fastapi_app/__init__.py:
from .models import CustomerRecord, ProductRecord
# Now callers can write:
from fastapi_app import CustomerRecord
# Instead of:
from fastapi_app.models import CustomerRecord
3. Initialise resources: connect to DB, load config (use sparingly)


## ABSOLUTE import: starts from the project root

from fastapi_app.models import CustomerRecord
from fastapi_app.routers.customers import router
RELATIVE import: starts from the current module's location (. = same folder)
from .models import CustomerRecord              # . = same package
from ..database import get_db_connection        # .. = parent package
When to use which:
- Absolute imports are clearer — use them in most cases
- Relative imports are useful inside packages to avoid circular imports
- FastAPI routers typically use relative imports internally


In [ ]:
def demonstrate_module_package_structure():
    """
    Print the project structure with module/package annotations.

    This function explains the structure of this project visually.
    """

    structure = """
PROJECT STRUCTURE — module01/
│
├── modules/                     ← NOT a package (no __init__.py)
│   ├── day01_setup_variables.py   ← module
│   ├── day02_conditions_loops.py  ← module
│   └── ...
│
├── notebooks/                   ← NOT a package (no __init__.py)
│   ├── day01_setup_variables.ipynb
│   └── ...
│
├── fastapi_app/                 ← PACKAGE (__init__.py present)
│   ├── __init__.py              ← marks this as a package
│   ├── main.py                  ← module (FastAPI app entry point)
│   ├── models.py                ← module (Pydantic models)
│   ├── database.py              ← module (DB connection)
│   └── routers/                 ← SUB-PACKAGE
│       ├── __init__.py          ← marks routers/ as a sub-package
│       ├── customers.py         ← module (customer API routes)
│       └── products.py          ← module (product API routes)
│
├── data/                        ← data directory (not a package)
│   ├── datasets/
│   │   ├── customers.csv
│   │   ├── products.csv
│   │   └── ...
│   └── knowledge_base/
│       └── semantic_products.json
│
└── prompts/                     ← prompt files (not a package)
    ├── system_prompt.txt
    └── few_shot_examples.json

IMPORT EXAMPLES (run from module01/ directory):
  from fastapi_app.models import CustomerRecord        ← absolute
  from fastapi_app.routers import customers            ← sub-package
  import modules.day01_setup_variables as day01        ← module (no __init__)
    """

    print(structure)




## PART B — requests LIBRARY (Synchronous HTTP Client)

═════════════════════════════════════════════════════════════════════════════
EVERY LLM API CALL IS AN HTTP POST REQUEST.
When you call openai.chat.completions.create() — it's sending an HTTP POST.
The requests library lets you make those calls directly.
requests vs httpx:
┌─────────────────┬─────────────────────────────────────────┐
│ requests        │ Synchronous — blocks while waiting      │
│                 │ Simple scripts, CLI tools, notebooks    │
│                 │ Cannot be used inside async functions   │
├─────────────────┼─────────────────────────────────────────┤
│ httpx (async)   │ Asynchronous — yields to event loop     │
│                 │ FastAPI endpoints, async agents          │
│                 │ Must be used inside async functions      │
└─────────────────┴─────────────────────────────────────────┘
Rule: if you're inside `async def`, use httpx. Otherwise, requests is fine.


In [ ]:
def get_request_example(url: str, params: dict | None = None) -> dict | None:
    """
    Make a GET request and return the JSON response.

    GET requests retrieve data. Parameters go in the URL query string.
    Example URL: https://api.example.com/products?category=Electronics&limit=10

    Args:
        url   : The endpoint URL.
        params: Optional query parameters (added to URL automatically).

    Returns:
        Parsed JSON dict, or None if the request fails.
    """

    try:
        # requests.get() makes the HTTP GET call
        # params={"key": "value"} is automatically URL-encoded
        # timeout=10 prevents hanging forever (always set a timeout!)
        response = requests.get(url, params=params, timeout=10)

        # response.raise_for_status() raises HTTPError for 4xx and 5xx responses
        # This is the recommended way to check for HTTP errors
        response.raise_for_status()

        # response.json() parses the JSON body into a Python dict
        return response.json()

    except requests.exceptions.Timeout:
        log.error(f"Request timed out: {url}")
        return None

    except requests.exceptions.ConnectionError as e:
        log.error(f"Cannot connect to {url}: {e}")
        return None

    except requests.exceptions.HTTPError as e:
        log.error(f"HTTP error {e.response.status_code}: {url}")
        return None



In [ ]:
def call_openai_api(
    prompt        : str,
    model         : str = "gpt-4o",
    max_tokens    : int = 512,
    temperature   : float = 0.2,
    api_key       : str | None = None,
) -> dict | None:
    """
    Call the OpenAI Chat Completions API using requests.

    This is the raw HTTP version of what openai.chat.completions.create()
    does internally. Understanding this helps you:
    - Debug API issues (see the exact request and response)
    - Call any OpenAI-compatible API (Bedrock, Together AI, etc.)
    - Understand what SDK libraries are doing under the hood

    Args:
        prompt     : The user message to send.
        model      : OpenAI model identifier.
        max_tokens : Max tokens in the response.
        temperature: Sampling temperature.
        api_key    : API key (reads from OPENAI_API_KEY env var if None).

    Returns:
        The full API response dict, or None on error.
    """

    key = api_key or os.environ.get("OPENAI_API_KEY", "")
    if not key:
        log.warning("OPENAI_API_KEY not set — using mock response")
        return _mock_openai_response(prompt)

    url     = "https://api.openai.com/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {key}",
        "Content-Type" : "application/json",
    }

    # The request body (payload) — this is the OpenAI API contract
    payload = {
        "model"      : model,
        "messages"   : [{"role": "user", "content": prompt}],
        "max_tokens" : max_tokens,
        "temperature": temperature,
    }

    try:
        log.info(f"POST {url} | model={model}")
        start    = time.perf_counter()

        response = requests.post(
            url,
            headers = headers,
            json    = payload,    # requests serialises dict → JSON automatically
            timeout = 30,
        )

        elapsed = round((time.perf_counter() - start) * 1000)
        log.info(f"Response: HTTP {response.status_code} in {elapsed}ms")

        response.raise_for_status()
        return response.json()

    except requests.exceptions.HTTPError as e:
        # Parse the error body for a meaningful message
        try:
            error_body = e.response.json()
            log.error(f"OpenAI API error: {error_body.get('error', {}).get('message', str(e))}")
        except Exception:
            log.error(f"OpenAI API HTTP error: {e}")
        return None



In [ ]:
def extract_llm_text_from_response(api_response: dict) -> str | None:
    """
    Extract the text content from an OpenAI API response dict.

    OpenAI response structure:
    {
      "choices": [
        {
          "message": {
            "role": "assistant",
            "content": "The actual response text goes here"
          },
          "finish_reason": "stop"
        }
      ],
      "usage": {"prompt_tokens": 10, "completion_tokens": 50, "total_tokens": 60}
    }

    Args:
        api_response: The dict returned by call_openai_api().

    Returns:
        The response text string, or None if the structure is unexpected.
    """

    if not api_response:
        return None

    # Chain of .get() calls — safe extraction from nested dicts
    choices = api_response.get("choices")
    if not choices:
        log.warning("API response has no 'choices' field")
        return None

    first_choice = choices[0]  # take the first (and usually only) choice
    message      = first_choice.get("message", {})
    content      = message.get("content")

    if content is None:
        log.warning("API response has no 'content' field in message")

    return content



In [ ]:
def call_llm_and_validate_json(prompt: str) -> dict | None:
    """
    Call the LLM and parse its JSON response.

    This combines:
    1. Making the HTTP request with requests
    2. Extracting the text content
    3. Parsing the JSON from the LLM's text
    4. Validating with safe .get() access

    Args:
        prompt: The prompt instructing the LLM to return JSON.

    Returns:
        Parsed dict from the LLM's JSON response, or None on failure.
    """

    response = call_openai_api(prompt)
    text = extract_llm_text_from_response(response)

    if not text:
        return None

    try:
        # Strip markdown fences (LLMs often add them)
        cleaned = text.strip()
        if cleaned.startswith("```"):
            cleaned = cleaned.split("\n", 1)[-1].rsplit("```", 1)[0].strip()

        return json.loads(cleaned)
    except json.JSONDecodeError as e:
        log.error(f"LLM returned non-JSON text: {e}\nText: {text[:200]}")
        return None



In [ ]:
def _mock_openai_response(prompt: str) -> dict:
    """Return a mock OpenAI-format response for demo purposes."""
    return {
        "choices": [{
            "message": {
                "role"   : "assistant",
                "content": json.dumps({
                    "category"  : "TRACK_ORDER",
                    "confidence": "high",
                    "reason"    : "Customer asked about order status and delivery",
                })
            },
            "finish_reason": "stop",
        }],
        "usage": {"prompt_tokens": 50, "completion_tokens": 30, "total_tokens": 80},
    }




In [ ]:
if __name__ == "__main__":
    print("=" * 60)
    print("DAY 11 — Modules, Packages, and requests")
    print("=" * 60)

    # Part A: Structure
    print("\n[1] Module vs Package Project Structure")
    demonstrate_module_package_structure()

    # Part B: requests
    print("\n[2] GET request (public API — JSONPlaceholder)")
    result = get_request_example(
        "https://jsonplaceholder.typicode.com/todos/1"
    )
    if result:
        print(f"  Response: {result}")

    print("\n[3] GET with query params")
    result = get_request_example(
        "https://jsonplaceholder.typicode.com/posts",
        params={"userId": 1, "_limit": 2}
    )
    if result:
        print(f"  Got {len(result)} posts")
        print(f"  First post: '{result[0]['title'][:50]}...'")

    print("\n[4] LLM API call (mock if no OPENAI_API_KEY)")
    classification_prompt = """
    Classify the following customer message and return JSON ONLY:
    {"category": "TRACK_ORDER|BILLING|RETURNS|OTHER", "confidence": "high|medium|low"}
    
    Message: "Where is my order #3042? It's been 8 days."
    """
    parsed = call_llm_and_validate_json(classification_prompt)
    if parsed:
        print(f"  Category  : {parsed.get('category')}")
        print(f"  Confidence: {parsed.get('confidence')}")

    print("\n[5] requests vs httpx — summary")
    print("""
    requests.get(url)          → sync, blocks the thread
    requests.post(url, json={}) → sync, blocks the thread
    Use for: scripts, notebooks, CLI tools, testing

    httpx.AsyncClient.get(url) → async, yields to event loop
    httpx.AsyncClient.post(url) → async, yields to event loop
    Use for: FastAPI endpoints, async agents, high-concurrency services
    """)


## Run the demo

Run the cell below to execute the `if __name__ == '__main__':` block.


In [ ]:
# Execute the demo for this day
import runpy
runpy.run_path('modules/day11_modules_packages_requests.py', run_name='__main__')
